# Explain a vision transformer with TorchCAM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/frgfm/notebooks/blob/main/torch-cam/vision_transformers.ipynb)

A vision transformer represents an image as a sequence of patch tokens. This notebook explains a **pretrained torchvision ViT-B/16** in two ways:

- **LeGrad** averages positive attention gradients from the last four encoder blocks.
- **GradCAM** reshapes token activations and gradients into a spatial grid.

Both examples target the same predicted ImageNet class and display their maps on the exact crop seen by the model. They use different objectives and layers, so visual agreement is not a measure of explanation quality.

Python **3.11 or newer** is required. A GPU is optional: the notebook falls back to CPU. The first model load downloads about **330 MiB** of pretrained weights; subsequent runs reuse PyTorch's checkpoint cache. CPU explanation takes longer and needs enough RAM for the model and gradient graph.

## Install a reproducible development snapshot

LeGrad and `reshape_transform` were introduced in **TorchCAM 0.5.0**. This tutorial pins a later development snapshot containing subsequent CAM and metric fixes. Pinning the commit keeps the method implementation stable; the exact Python, Torch, and torchvision versions are printed in the next cell.

Run the setup cell once in a fresh kernel. If this upgrades packages already imported in your notebook session, restart the kernel before continuing. Use matching Torch and torchvision builds for your CPU or CUDA environment.

In [ ]:
%pip install -q "torchcam[scripts] @ git+https://github.com/frgfm/torch-cam.git@e311623a60474aeab359afee5c432cc97a27269e"

In [ ]:
import hashlib
import platform
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import torch
import torchvision
from PIL import Image
from torchvision.models import ViT_B_16_Weights, vit_b_16
from torchvision.transforms import functional as TF

import torchcam
from torchcam.methods import GradCAM, LeGrad
from torchcam.utils import overlay_mask

assert tuple(map(int, platform.python_version_tuple()[:2])) >= (3, 11)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Keep the CPU example responsive on shared notebook machines.
if device.type == "cpu":
    torch.set_num_threads(min(2, torch.get_num_threads()))

print(f"Python: {platform.python_version()}")
print(f"Torch: {torch.__version__}; torchvision: {torchvision.__version__}")
print(f"TorchCAM: {torchcam.__version__}")
print(f"Device: {device}; CUDA runtime: {torch.version.cuda}")
print("TorchCAM source: e311623a60474aeab359afee5c432cc97a27269e")

## Load the pretrained model and one image

Use an explicit weights enum rather than `DEFAULT`, which may change in a future torchvision release. `weights.transforms()` supplies the matching resize, center crop, normalization, and ImageNet class names. The example photograph comes from the [PyTorch Hub repository](https://github.com/pytorch/hub/blob/master/images/dog.jpg) and is cached locally.

In [ ]:
weights = ViT_B_16_Weights.IMAGENET1K_V1
model = vit_b_16(weights=weights).to(device).eval()
preprocess = weights.transforms()
categories = weights.meta["categories"]

image_url = "https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg"
cache_dir = Path.home() / ".cache" / "torchcam-notebooks"
cache_dir.mkdir(parents=True, exist_ok=True)
image_path = cache_dir / "dog.jpg"
if not image_path.exists():
    urlretrieve(image_url, image_path)
with Image.open(image_path) as source:
    image = source.convert("RGB")

print(f"Weights: {weights.name}")
print(f"Image source: {image_url}")
print(f"Image SHA256: {hashlib.sha256(image_path.read_bytes()).hexdigest()}")
print(preprocess)

The CAM describes the **224 × 224 center crop**, not the original photograph. Overlaying it directly on the uncropped image would stretch it onto the wrong coordinates. Reproduce the spatial part of `weights.transforms()` for the background image, and check that it matches the normalized input.

In [ ]:
input_tensor = preprocess(image).unsqueeze(0).to(device)
aligned_image = TF.center_crop(
    TF.resize(
        image,
        preprocess.resize_size,
        interpolation=preprocess.interpolation,
        antialias=preprocess.antialias,
    ),
    preprocess.crop_size,
)
mean = torch.tensor(preprocess.mean).view(3, 1, 1)
std = torch.tensor(preprocess.std).view(3, 1, 1)
assert torch.allclose(
    TF.to_tensor(aligned_image), input_tensor[0].cpu() * std + mean, atol=1e-6
)

print(f"Original: {image.size}; model crop: {aligned_image.size}")
print(f"Input shape: {tuple(input_tensor.shape)}")
plt.figure(figsize=(4, 4))
plt.imshow(aligned_image)
plt.title("The model's input crop")
plt.axis("off")
plt.show()

In [ ]:
# Ordinary prediction can run without gradients before any CAM hooks exist.
with torch.no_grad():
    prediction_scores = model(input_tensor)
assert prediction_scores.shape == (1, len(categories))
probabilities = prediction_scores.softmax(dim=1)[0].cpu()
target_class = int(probabilities.argmax())

print(f"Output shape: {tuple(prediction_scores.shape)}")
print(f"Explanation target: {target_class} — {categories[target_class]}")
top_k = min(5, probabilities.numel())
for index in probabilities.topk(top_k).indices.tolist():
    print(f"{categories[index]:>20}: {probabilities[index]:.1%}")
del prediction_scores

## LeGrad: attention gradients across encoder blocks

Pass **complete transformer blocks**, not their normalization or attention submodules. Each selected block must return `(batch, tokens, embedding)` and expose a direct `self_attention` child implemented by batch-first `torch.nn.MultiheadAttention`, with shared-dimension Q/K/V projections and no added attention tokens. Evaluation mode disables attention dropout.

For torchvision ViT, the built-in intermediate score projection averages tokens, applies `model.encoder.ln`, and then `model.heads`. LeGrad differentiates that **layer-specific class score** with respect to post-softmax attention probabilities. It averages positive gradients over heads and query tokens, removes the class-token key, and averages the patch maps over selected blocks. It does not multiply attention values by their gradients or differentiate the final class-token logit.

This adapter targets torchvision's `VisionTransformer` contract. **timm, Swin, and OpenCLIP models are not automatically supported** just because they are transformers; models with another attention implementation require a compatible adapter. Other blocks satisfying the contract also need an explicit `score_projection` when they lack torchvision's encoder/head structure.

Each extractor owns hooks for one forward pass. Keep gradient tracking enabled while it is active; `model.eval()` is compatible with autograd. Do not put either explanation inside `torch.no_grad()` or `torch.inference_mode()`.

In [ ]:
grid_shape = (model.image_size // model.patch_size,) * 2
legrad_blocks = list(model.encoder.layers)[-4:]
module_names = {module: name for name, module in model.named_modules()}
print("LeGrad blocks:", [module_names[block] for block in legrad_blocks])
print("Patch grid:", grid_shape)

with torch.enable_grad(), LeGrad(
    model,
    target_layer=legrad_blocks,
    prefix_tokens=1,
    grid_shape=grid_shape,
) as extractor:
    scores = model(input_tensor)
    assert scores.shape == (1, len(categories))
    legrad_map = extractor(target_class)[0].detach().cpu()
del scores

## GradCAM: reshape patch tokens into a grid

GradCAM expects `(batch, channels, height, width)` activations. ViT normalization layers instead return `(batch, tokens, embedding)`. The reshape callback drops the one prefix/class token, checks the patch count, reshapes using the model's image and patch sizes, and moves embedding channels first. TorchCAM applies the same callback to the gradients.

Use an explicit layer whose patch tokens can still influence the final class token: here, the first normalization in the **penultimate encoder block**, `model.encoder.layers[-2].ln_1`, before self-attention. The patch tokens at the final encoder output normalization are a poor choice for this classifier: its head reads only the class token, so those final patch tokens have no gradient path to the logit.

This GradCAM explains the **final output logit**, unlike LeGrad's intermediate score projection. It runs a fresh forward pass after LeGrad's context removes its hooks.

In [ ]:
def tokens_to_grid(tokens):
    if tokens.ndim != 3:
        raise ValueError("Expected (batch, tokens, embedding) activations")
    patch_tokens = tokens[:, 1:, :]  # torchvision ViT has one class token
    height, width = grid_shape
    if patch_tokens.shape[1] != height * width:
        raise ValueError("Patch-token count does not match the model's spatial grid")
    return patch_tokens.reshape(
        tokens.shape[0], height, width, tokens.shape[-1]
    ).permute(0, 3, 1, 2)


gradcam_layer = model.encoder.layers[-2].ln_1
print("GradCAM layer:", module_names[gradcam_layer])

In [ ]:
with torch.enable_grad(), GradCAM(
    model,
    target_layer=gradcam_layer,
    reshape_transform=tokens_to_grid,
) as extractor:
    scores = model(input_tensor)
    assert scores.shape == (1, len(categories))
    gradcam_map = extractor(target_class, scores=scores)[0].detach().cpu()
del scores

## Check and compare the maps

Both maps should contain finite values and spatial variation. These checks catch a disconnected target layer or a broken token reshape; they do not establish faithfulness. Each method normalizes its own map independently to `[0, 1]`, so overlay intensity cannot compare attribution strength across methods.

In [ ]:
maps = {"LeGrad — last four blocks": legrad_map, "GradCAM — token activations": gradcam_map}
for name, cam in maps.items():
    assert tuple(cam.shape) == (1, *grid_shape), (name, cam.shape)
    assert torch.isfinite(cam).all(), f"{name} contains non-finite values"
    assert float(cam.max() - cam.min()) > 1e-6, f"{name} is blank"
    assert float(cam.min()) >= -1e-6 and float(cam.max()) <= 1 + 1e-6
    print(f"{name}: shape={tuple(cam.shape)}, range=[{cam.min():.3f}, {cam.max():.3f}]")

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(aligned_image)
axes[0].set_title(f"Input: {categories[target_class]}")
for axis, (name, cam) in zip(axes[1:], maps.items(), strict=True):
    # Keep a floating-point mask so overlay_mask receives values in [0, 1].
    mask = TF.to_pil_image(cam[0], mode="F")
    axis.imshow(overlay_mask(aligned_image, mask, alpha=0.5))
    axis.set_title(name)
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.show()

The displayed maps have a native **14 × 14 patch grid**; resizing them for an overlay does not add spatial evidence. Highlighted regions show sensitivity to the selected class under each method's objective. They are not segmentation masks or proof that an object caused the prediction.

To explore another class, set `target_class` to its ImageNet index and rerun both explanation cells and the comparison. Always rerun each forward pass inside a new extractor context. For a quantitative comparison, evaluate perturbation/faithfulness metrics rather than selecting the map that looks most plausible.

See the [TorchCAM transformer guide](https://frgfm.github.io/torch-cam/getting-started/advanced-usage/#vision-transformers-and-other-non-cnn-models) and [method API](https://frgfm.github.io/torch-cam/reference/methods/) for supported contracts and options.